# Week 2: Metric Engineering (Momentum Score)
This notebook engineered our core proprietary metric: the **Momentum Score**, and computes supporting indicators like YoY growth, rolling average ticket sizes, and volume share shifts.

In [1]:
import os
import pandas as pd
import numpy as np

PROCESSED_DIR = '../data/processed'
df = pd.read_csv(os.path.join(PROCESSED_DIR, 'master.csv'))
df['month'] = pd.to_datetime(df['month'])

In [2]:
df = df.sort_values(by=['category', 'month']).reset_index(drop=True)
df['mom_pct'] = df.groupby('category')['volume'].pct_change()

df['overall_volume'] = df.groupby('month')['volume'].transform('sum')
overall_df = df[['month', 'overall_volume']].drop_duplicates().sort_values('month').reset_index(drop=True)

for w in [3, 6, 12]:
    df[f'volume_lag_{w}m'] = df.groupby('category')['volume'].shift(w)
    df[f'category_growth_{w}m'] = (df['volume'] / df[f'volume_lag_{w}m']) - 1
    
    overall_df[f'overall_volume_lag_{w}m'] = overall_df['overall_volume'].shift(w)
    overall_df[f'overall_growth_{w}m'] = (overall_df['overall_volume'] / overall_df[f'overall_volume_lag_{w}m']) - 1
    
    df = df.merge(overall_df[['month', f'overall_growth_{w}m']], on='month', how='left')
    
    df[f'momentum_score_{w}m'] = df[f'category_growth_{w}m'] / df[f'overall_growth_{w}m']
    overall_df = overall_df.drop(columns=[f'overall_volume_lag_{w}m'])

df['momentum_score'] = df['momentum_score_3m']

df['volume_lag_12m_yoy'] = df.groupby('category')['volume'].shift(12)
df['yoy_growth'] = (df['volume'] / df['volume_lag_12m_yoy']) - 1

df['avg_ticket_rolling_3m'] = df.groupby('category')['avg_ticket'].transform(
    lambda x: x.rolling(3, min_periods=1).mean()
)

df['rolling_12m_avg_vol'] = df.groupby('category')['volume'].transform(
    lambda x: x.rolling(12, min_periods=1).mean()
)
df['seasonality_index'] = df['volume'] / df['rolling_12m_avg_vol']

df['volume_share_pct'] = (df['volume'] / df['overall_volume']) * 100
df['rolling_3m_cat_vol'] = df.groupby('category')['volume'].transform(lambda x: x.rolling(3, min_periods=1).sum())
df['rolling_3m_overall_vol'] = df.groupby('category')['overall_volume'].transform(lambda x: x.rolling(3, min_periods=1).sum())
df['volume_share_3m'] = (df['rolling_3m_cat_vol'] / df['rolling_3m_overall_vol']) * 100
df['volume_share_3m_lag_12m'] = df.groupby('category')['volume_share_3m'].shift(12)
df['volume_share_shift'] = df['volume_share_3m'] - df['volume_share_3m_lag_12m']

drop_cols = [c for c in df.columns if 'lag_' in c or 'growth_' in c or 'rolling_3m_' in c]
df_final = df.drop(columns=drop_cols)

enriched_filepath = os.path.join(PROCESSED_DIR, 'master_enriched.csv')
df_final.to_csv(enriched_filepath, index=False)
print(f'Created master_enriched.csv with columns: {df_final.columns.tolist()}')
df_final.tail(10)

Created master_enriched.csv with columns: ['month', 'category', 'volume', 'value_cr', 'avg_ticket', 'mom_pct', 'overall_volume', 'momentum_score_3m', 'momentum_score_6m', 'momentum_score_12m', 'momentum_score', 'yoy_growth', 'avg_ticket_rolling_3m', 'rolling_12m_avg_vol', 'seasonality_index', 'volume_share_pct', 'volume_share_3m', 'volume_share_shift']


,month,category,volume,value_cr,avg_ticket,mom_pct,overall_volume,momentum_score_3m,momentum_score_6m,momentum_score_12m,momentum_score,yoy_growth,avg_ticket_rolling_3m,rolling_12m_avg_vol,seasonality_index,volume_share_pct,volume_share_3m,volume_share_shift
1192,2025-08-01,Variety Stores,67180000,3392.94,505.052099,0.115391,10325270000,1.595529,0.835723,0.892403,1.595529,0.418497,523.629193,5.683083e+07,1.182105,0.650637,0.621133,-0.024212
1193,2025-09-01,Variety Stores,66379999,3546.90,534.332638,-0.011908,12366969997,0.417427,0.263122,0.550298,0.417427,0.384070,520.574701,5.836583e+07,1.137309,0.536752,0.590261,-0.068110
1194,2025-10-01,Variety Stores,71730000,4055.70,565.411962,0.080597,13027560000,0.670194,0.464742,0.627428,0.670194,0.448506,534.932233,6.021667e+07,1.191198,0.550602,0.574723,-0.086359
1195,2025-11-01,Variety Stores,68980000,3693.49,535.443607,-0.038338,12840490000,0.109991,0.405563,0.498806,0.109991,0.341501,545.062736,6.168000e+07,1.118353,0.537207,0.541624,-0.120076
1196,2025-12-01,Variety Stores,68820000,3557.70,516.957280,-0.002320,13482380000,0.407551,0.399307,0.353081,0.407551,0.218053,539.270949,6.270667e+07,1.097491,0.510444,0.532472,-0.135948
1197,2026-01-01,Variety Stores,67820000,3555.65,524.277499,-0.014531,13608100000,-1.223226,0.368306,0.389834,-1.223226,0.290580,525.559462,6.397917e+07,1.060033,0.498380,0.514939,-0.160621
1198,2026-02-01,Variety Stores,60770000,3225.38,530.752016,-0.103952,12795780000,34.181955,-0.398780,0.118745,34.181955,0.058342,523.995598,6.425833e+07,0.945714,0.474922,0.494932,-0.178716
1199,2026-03-01,Variety Stores,67040000,3425.60,510.978520,0.103176,14180459999,-0.499536,0.067804,0.198946,-0.499536,0.145787,522.002678,6.496917e+07,1.031874,0.472763,0.482033,-0.203952
1200,2026-04-01,Variety Stores,70940000,3569.38,503.154779,0.058174,14110220000,1.246771,-0.132525,0.334615,1.246771,0.172562,514.961772,6.583917e+07,1.077474,0.502756,0.483736,-0.192939
1201,2026-05-01,Variety Stores,77400000,3738.98,483.072351,0.091063,14791149999,1.754876,0.803506,0.513587,1.754876,0.268852,499.068550,6.720583e+07,1.151686,0.523286,0.499932,-0.161828
